# Reproducing a club's eligibility from the raw match record

This notebook re-derives one club's **Match Participation** from the archived Rugby Xplorer match records using only the Python standard library, with no code from the `@zmnatz/usa-club-rugby-data` package, then compares the result with the published file the website reads. If the two agree, the published number can be reproduced by anyone from the raw record and the written rules.

The pipeline it walks, left to right:

```
Rugby Xplorer CMS  ->  archive/  (raw)  ->  public/data/  (published)  ->  verdicts
   `snapshot` command   this notebook recounts this step   library: clubEligibility()
```

Set `CLUB` below to any club key from `public/data/index.json` (the page path, e.g. `rocky-gorge`). Run from the repository root; if `archive/` is not on disk the files are downloaded from GitHub.

In [ ]:
import json, os, re, urllib.request
from collections import defaultdict

CLUB = "rocky-gorge"
REPO_RAW = "https://raw.githubusercontent.com/zmnatz/usa-club-rugby-eligibility/main"

def read(path):
    if os.path.exists(path):
        return open(path, encoding="utf8").read()
    with urllib.request.urlopen(f"{REPO_RAW}/{path}") as response:
        return response.read().decode("utf8")

index = json.loads(read("public/data/index.json"))
club = next(c for c in index["clubs"] if c["key"] == CLUB)
published = json.loads(read(f"public/data/clubs/{CLUB}.json"))
season = index["season"]
print(club["name"], season, [s["teamName"] for s in club["sides"]])
print("published at", index["generatedAt"])

## 1. The inputs

Two archived files for the season: every fixture, and every played match's full CMS record (lineup, substitutes, coaches and commentary). Both are plain JSON, one entry per line.

In [ ]:
season_file = re.sub(r"[^0-9A-Za-z]+", "-", season) + ".json"
fixtures = json.loads(read(f"archive/fixtures/{season_file}"))["fixtures"]
matches = json.loads(read(f"archive/matches/{season_file}"))["matches"]
print(len(fixtures), "fixtures,", sum(m is not None for m in matches.values()), "match records")

## 2. The rules (config)

Competitions whose name contains one of these fragments (case-insensitive) are not NCS Qualifying Matches. These are the library's `DEFAULT_RULES`; run `usa-club-rugby-data rules` to print them.

In [ ]:
NON_QUALIFYING = ["Super Regional", "ASR ", "National Championships", "7s", "7's", "Sevens",
                  "Friendly", "Friendlies", "Crossover", "Social", "Festival", "Qualifier Rosters"]
MINIMUM_MATCHES = 2

def qualifying(comp):
    name = comp.lower()
    return not any(t.lower() in name for t in NON_QUALIFYING)

## 3. Counting a player's matches

For each played fixture of the club's sides, from the raw record:

- a **starter** is anyone in `players` (Played);
- a **substitute** counts only if the commentary shows them coming on (`on: <name>`); being listed on the bench doesn't count (Reg 3.4.2(e));
- a **forfeit** counts for every rostered player on the *non-forfeiting* side, starters and replacements alike, and for no one on the forfeiting side (Reg 3.4(h));
- coaches never count, and a hidden profile ("Name Withheld") is not a player.

In [ ]:
team_names = {s["teamName"] for s in club["sides"]}
pkey = lambda pid: pid.split("__")[0]

def subs_who_came_on(match):
    lineup = match["players"] + match["substitutes"]
    came_on = set()
    for event in match["commentary"]:
        if "substitut" not in event["type"].lower():
            continue
        m = re.search(r"(?:off:\s*(.+?),\s*)?on:\s*(.+?)\s*$", event["comment"], re.I)
        if not m:
            continue
        off = m.group(1).strip().lower() if m.group(1) else None
        sides = {p["isHome"] for p in lineup if p["name"].lower() == off}
        home = next(iter(sides)) if len(sides) == 1 else None
        for sub in match["substitutes"]:
            if (pkey(sub["id"]) not in came_on and sub["name"].lower() == m.group(2).strip().lower()
                    and (home is None or sub["isHome"] == home)):
                came_on.add(pkey(sub["id"]))
                break
    return came_on

def forfeit_winner(fx):
    h, a = fx["homeTeam"]["score"], fx["awayTeam"]["score"]
    if not h or not a or int(h) == int(a):
        return None
    return fx["homeTeam"]["name"] if int(h) > int(a) else fx["awayTeam"]["name"]

counts = defaultdict(lambda: {"played": 0, "substitute": 0})
for fid in published["resultIds"]:
    fx, match = fixtures[fid], matches.get(fid)
    if match is None:
        continue
    came_on = subs_who_came_on(match)
    starters = {pkey(p["id"]) for p in match["players"]}
    seen = set()
    for p in match["players"] + match["substitutes"]:
        key = pkey(p["id"])
        if key in seen or p["name"].strip().lower() == "name withheld":
            continue
        seen.add(key)
        team = fx["homeTeam"]["name"] if p["isHome"] else fx["awayTeam"]["name"]
        if team not in team_names:
            continue
        forfeit = fx["status"] == "Forfeit"
        if forfeit and forfeit_winner(fx) != team:
            continue
        entry = counts[(team, fx["compName"], key)]
        if key in starters:
            entry["played"] += 1
        elif forfeit or key in came_on:
            entry["substitute"] += 1
print(len(counts), "player/team/competition rows recounted")

## 4. Compare with the published file

The published file was calculated by the library. Every row should match.

In [ ]:
ours = {k: v for k, v in counts.items() if v["played"] + v["substitute"] > 0}
theirs = {(r["teamName"], r["competition"], r["playerId"]): {"played": r["played"], "substitute": r["substitute"]}
          for r in published["rows"] if r["played"] + r["substitute"] > 0}
only_ours, only_theirs = ours.keys() - theirs.keys(), theirs.keys() - ours.keys()
differ = [k for k in ours.keys() & theirs.keys() if ours[k] != theirs[k]]
print(f"rows: notebook {len(ours)}, published {len(theirs)}")
print(f"only in notebook: {len(only_ours)}, only in published: {len(only_theirs)}, differing counts: {len(differ)}")
for k in list(differ)[:10]:
    print("  ", k, "notebook", ours[k], "published", theirs[k])

Disagreements are possible only where the library applies a rule this notebook leaves out: its surname fallback for a substitution event that names a player slightly differently from the lineup (`playerHistory.ts`, `substitutesWhoCameOn`). Anything beyond a handful of rows means one of the two implementations has drifted from the written rules.

## 5. From counts to a verdict

For a club that fields one side, Reg 3.4.2(e) asks for `MINIMUM_MATCHES` NCS Qualifying Matches. Clubs with several sides add Reg 3.5, which the library implements (see `docs/adr/000-rules-carried-from-rocky-gorge.md` and the regulation table in `eligibility-regulation.test.ts`). Here is the single-threshold check, summed over the club's sides:

In [ ]:
totals = defaultdict(int)
names = {r["playerId"]: f'{r["firstName"]} {r["lastName"]}' for r in published["rows"]}
for (team, comp, pid), v in ours.items():
    if qualifying(comp):
        totals[pid] += v["played"] + v["substitute"]
meet = sorted(((n, names.get(p, p)) for p, n in totals.items()), reverse=True)
print(sum(1 for n, _ in meet if n >= MINIMUM_MATCHES), "of", len(meet), f"players have {MINIMUM_MATCHES}+ qualifying matches across the club")
for n, name in meet[:10]:
    print(f"{n:>3}  {name}")

## 6. The library's verdicts, for comparison

```js
import { createPublishedClient } from '@zmnatz/usa-club-rugby-data';
const client = createPublishedClient();               // or { baseUrl: 'https://...' }
const players = await client.players('rocky-gorge');  // one verdict per player per side
```

Each verdict carries `participation`, `eligible`, `reasons` and `pending`. To check that the published files themselves are the product of the archive, run `bun run verify` (or `npx @zmnatz/usa-club-rugby-data verify`), which recalculates all of `public/data/` from `archive/` and fails on any difference.